WSI 폴더에서 일괄 배치 처리

In [ ]:
import subprocess

WSI_FLAT_DIR = './wsis'
RESULT_DIR = './working/cptac_processed_uni_v2'
PATCH_ENCODER = 'uni_v2'

# 쉼표(,)를 제거하고 각 줄 끝에 공백을 추가하여 하나의 문자열로 만듭니다.
command = (
    "python ./encoders/run_batch_of_slides.py "
    "--task all "
    f"--wsi_dir {WSI_FLAT_DIR} "
    f"--job_dir {RESULT_DIR} "
    f"--patch_encoder {PATCH_ENCODER} "
    "--batch_size 32 "
    "--segmenter grandqc "
    "--remove_penmarks "
    "--mag 20 --patch_size 256"
)

subprocess.run(
    command,
    shell=True,
    check=True
)

Traceback (most recent call last):
  File "/home/team1/jsy/team_project/VisionGen-COAD/jsy/./utils/run_batch_of_slides.py", line 16, in <module>
    from trident import Processor 
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/team1/jsy/team_project/TRIDENT/trident/__init__.py", line 8, in <module>
    from trident.wsi_objects.OpenSlideWSI import OpenSlideWSI
  File "/home/team1/jsy/team_project/TRIDENT/trident/wsi_objects/OpenSlideWSI.py", line 7, in <module>
    from trident.wsi_objects.WSI import WSI, ReadMode
  File "/home/team1/jsy/team_project/TRIDENT/trident/wsi_objects/WSI.py", line 10, in <module>
    from trident.segmentation_models.load import SegmentationModel
  File "/home/team1/jsy/team_project/TRIDENT/trident/segmentation_models/__init__.py", line 2, in <module>
    from trident.segmentation_models.load import (
  File "/home/team1/jsy/team_project/TRIDENT/trident/segmentation_models/load.py", line 9, in <module>
    from trident.IO import get_dir, get_weights_path, has

CalledProcessError: Command 'python ./utils/run_batch_of_slides.py --task all --wsi_dir ./wsis --job_dir ./working/cptac_processed_uni_v2 --patch_encoder uni_v2 --batch_size 32 --segmenter grandqc --remove_penmarks --mag 20 --patch_size 256' returned non-zero exit status 1.

패치개수 비교 : 좌표 벡터와 특성 벡터의 차원 비교

In [17]:
import h5py
import os

coords_path = "./symlink_data/trident_processed/20.0x_256px_0px_overlap/patches_uni_v2"
feats_path = "./symlink_data/trident_processed/20.0x_256px_0px_overlap/features_gigapath"
coords_files = sorted(os.listdir(coords_path))
feats_files = sorted(os.listdir(feats_path))
count = 0
for coord, feat in zip(coords_files, feats_files):
    with h5py.File(os.path.join(coords_path, coord), 'r') as f_c:
        n_coords = f_c['coords'].shape
        coords_config = dict(f_c['coords'].attrs)
    with h5py.File(os.path.join(feats_path, feat), 'r') as f_f:
        n_feats = f_f['features'].shape
    print(f'벡터 차원 {coord}/{feat} = {n_coords}/{n_feats}')
    print('coords config :', coords_config)
    if n_coords[0] != n_feats[0]:
        count += 1
print(len(coords_files), "개 중,", count, "개 일치하지 않음")
# # 무작위로 슬라이드 하나를 골라서 확인
# with h5py.File('symlink_data/trident_processed/20.0x_256px_0px_overlap/patches_gigapath/TCGA-3L-AA1B-01Z-00-DX1_patches.h5', 'r') as f_c:
#     n_coords = f_c['coords'].shape[0]

# with h5py.File('symlink_data/trident_processed/20.0x_256px_0px_overlap/patches_gigapath/TCGA-3L-AA1B-01Z-00-DX1_patches.h5', 'r') as f_f:
#     n_feats = f_f['features'].shape[0]

# print(f"Coords: {n_coords}, Features: {n_feats}")
# 결과가 같으면 90% 이상 성공입니다.

벡터 차원 TCGA-3L-AA1B-01Z-00-DX1_patches.h5/TCGA-3L-AA1B-01Z-00-DX1.h5 = (15070, 2)/(15070, 1536)
coords config : {'level0_height': np.int64(74462), 'level0_magnification': np.int64(40), 'level0_width': np.int64(95615), 'name': 'TCGA-3L-AA1B-01Z-00-DX1', 'overlap': np.int64(0), 'patch_size': np.int64(256), 'patch_size_level0': np.float64(512.0), 'savetodir': './trident_processed/20.0x_256px_0px_overlap', 'target_magnification': np.float64(20.0)}
벡터 차원 TCGA-3L-AA1B-01Z-00-DX2_patches.h5/TCGA-3L-AA1B-01Z-00-DX2.h5 = (8521, 2)/(8521, 1536)
coords config : {'level0_height': np.int64(52434), 'level0_magnification': np.int64(40), 'level0_width': np.int64(87647), 'name': 'TCGA-3L-AA1B-01Z-00-DX2', 'overlap': np.int64(0), 'patch_size': np.int64(256), 'patch_size_level0': np.float64(512.0), 'savetodir': './trident_processed/20.0x_256px_0px_overlap', 'target_magnification': np.float64(20.0)}
벡터 차원 TCGA-4N-A93T-01Z-00-DX1_patches.h5/TCGA-4N-A93T-01Z-00-DX1.h5 = (16392, 2)/(16392, 1536)
coords config

In [ ]:
with h5py.File('/home/team/projects/team_REDCAB/team_project/trident_processed/20.0x_256px_0px_overlap/patches/TCGA-AA-3833-01Z-00-DX1_patches.h5', 'r') as f:
    print("UNI Config:", dict(f['coords'].attrs))

with h5py.File('/home/team/projects/team_REDCAB/team_project/trident_processed/20.0x_256px_0px_overlap/features_gigapath/TCGA-AA-3833-01Z-00-DX1.h5', 'r') as f:
    # GigaPath 특성 파일에도 추출 당시의 좌표 설정이 기록되어 있을 수 있습니다.
    if 'coords' in f:
        print("GigaPath Config:", dict(f['coords'].attrs))

In [3]:
import os
import pandas as pd

SEED = 42

df = pd.read_csv('./labels/cptac_coad_patient_msi.csv')
df['msi'] = df['type'].map({'MSS':0, 'MSIMUT':1})

df_train_val = df

df['fold_0'] = 'test'

label_path = os.path.join('labels', f'clinical_data_seed_{SEED}.csv')
os.makedirs('labels', exist_ok=True)
df.to_csv(label_path, index=False)